# Comparer les branches de detection

Ce notebook lance les memes cas de test centralises contre les branches activees : `generic`, `spacy`, `gliner`, `regex`.

Les interrupteurs de branches sont dans la premiere cellule de code.


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ENABLE_GENERIC_BRANCH = True
ENABLE_SPACY_BRANCH = True
ENABLE_GLINER_BRANCH = True
ENABLE_REGEX_BRANCH = True

MODEL_CACHE_DIR = r"D:\Workspaces\ModelCache"
MODEL_STORE_DIR = r"D:\Workspaces\modelStore"
SPACY_MODEL = rf"{MODEL_STORE_DIR}\fr_core_news_md"
SPACY_SYNONYMS_PATH = ROOT / "configs" / "spacy_synonyms.csv"
GLINER_MODEL = rf"{MODEL_STORE_DIR}\gliner_multi-v2.1"
GLINER_SOURCE_MODEL = "urchade/gliner_multi-v2.1"
GLINER_THRESHOLD = 0.50
GLINER_LOCAL_FILES_ONLY = True
GLINER_LABELS = [
    "donnee de sante",
    "maladie",
    "pathologie",
    "etat de sante",
    "condition medicale",
    "probleme de sante",
    "opinion politique",
    "conviction religieuse",
    "appartenance syndicale",
    "orientation sexuelle",
    "origine ethnique",
    "donnee genetique",
    "donnee biometrique",
    "clause beneficiaire imprecise",
    "conseil non professionnel",
    "promesse de performance",
]

pd.set_option("display.max_colwidth", 180)

os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = rf"{MODEL_CACHE_DIR}\transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = rf"{MODEL_CACHE_DIR}\hub"
os.environ["COMPLIANCE_NLP_MODEL_CACHE"] = MODEL_CACHE_DIR
os.environ["COMPLIANCE_NLP_MODEL_STORE"] = MODEL_STORE_DIR

enabled_branches = tuple(
    branch
    for branch, enabled in {
        "generic": ENABLE_GENERIC_BRANCH,
        "spacy": ENABLE_SPACY_BRANCH,
        "gliner": ENABLE_GLINER_BRANCH,
        "regex": ENABLE_REGEX_BRANCH,
    }.items()
    if enabled
)

params = {
    "enabled_branches": enabled_branches,
    "model_cache_dir": MODEL_CACHE_DIR,
    "model_store_dir": MODEL_STORE_DIR,
    "spacy_model": SPACY_MODEL,
    "spacy_synonyms_path": str(SPACY_SYNONYMS_PATH),
    "gliner_model": GLINER_MODEL,
    "gliner_threshold": GLINER_THRESHOLD,
    "gliner_local_files_only": GLINER_LOCAL_FILES_ONLY,
    "gliner_labels": GLINER_LABELS,
}
params


In [ ]:
from compliance_nlp import (
    load_generic_detection_rules,
    load_section_definitions,
    load_whitelist_terms,
)
from compliance_nlp.pipeline import analyze_text

sections = load_section_definitions(ROOT / "configs" / "sections.csv")
generic_rules = load_generic_detection_rules(ROOT / "configs" / "generic_detection_rules.csv")
whitelist_terms = load_whitelist_terms(ROOT / "configs" / "article9_whitelist.csv")

referentials_df = pd.DataFrame([
    {"referentiel": "sections", "count": len(sections)},
    {"referentiel": "generic_rules", "count": len(generic_rules)},
    {"referentiel": "whitelist_terms", "count": len(whitelist_terms)},
    {"referentiel": "gliner_labels", "count": len(GLINER_LABELS)},
])
referentials_df


In [ ]:
TEST_CASES_PATH = ROOT / "configs" / "test_cases.csv"
TEST_CASE_COLUMNS = ["text", "control_family", "comment"]

test_cases = pd.read_csv(TEST_CASES_PATH, keep_default_na=False)
missing_columns = set(TEST_CASE_COLUMNS) - set(test_cases.columns)
if missing_columns:
    raise ValueError(f"Colonnes manquantes dans {TEST_CASES_PATH}: {sorted(missing_columns)}")

test_cases = test_cases[TEST_CASE_COLUMNS].copy()
test_cases["case_id"] = [f"case_{index + 1:02d}" for index in range(len(test_cases))]
test_cases = test_cases[["case_id", "text", "control_family", "comment"]]
test_cases


In [ ]:
def finding_key(finding):
    return finding.rule_id or finding.code


def run_detector(text: str, case_id: str):
    return analyze_text(
        document_name=case_id,
        source_path="notebook",
        extracted_text=text,
        generic_rules=generic_rules,
        section_definitions=sections,
        whitelist_terms=whitelist_terms,
        enabled_branches=enabled_branches,
        spacy_model=SPACY_MODEL,
        spacy_synonyms_path=str(SPACY_SYNONYMS_PATH),
        gliner_model=GLINER_MODEL,
        gliner_cache_dir=MODEL_CACHE_DIR,
        gliner_source_model=GLINER_SOURCE_MODEL,
        gliner_labels=GLINER_LABELS,
        gliner_threshold=GLINER_THRESHOLD,
        gliner_local_files_only=GLINER_LOCAL_FILES_ONLY,
    )


def empty_row(case, engine):
    return {
        "case_id": case["case_id"],
        "control_family": case["control_family"],
        "comment": case["comment"],
        "detection_engine": engine,
        "predicted_key": None,
        "code": None,
        "rule_id": None,
        "rule_scope": None,
        "regulatory_family": None,
        "section": None,
        "category": None,
        "matched_term": None,
        "detection_type": None,
        "score": None,
        "branch_score": None,
        "generic_score": None,
        "spacy_score": None,
        "gliner_score": None,
        "regex_score": None,
        "severity": None,
        "evidence": None,
    }


def findings_to_rows(case, analysis):
    findings = analysis.findings
    if not findings:
        return [empty_row(case, "none")]

    rows = []
    for finding in findings:
        rows.append({
            "case_id": case["case_id"],
            "control_family": case["control_family"],
            "comment": case["comment"],
            "detection_engine": finding.detection_engine,
            "predicted_key": finding_key(finding),
            "code": finding.code,
            "rule_id": finding.rule_id,
            "rule_scope": finding.rule_scope,
            "regulatory_family": finding.regulatory_family,
            "section": finding.section,
            "category": finding.category,
            "matched_term": finding.matched_term,
            "detection_type": finding.detection_type,
            "score": finding.score,
            "branch_score": finding.branch_score,
            "generic_score": finding.generic_score,
            "spacy_score": finding.spacy_score,
            "gliner_score": finding.gliner_score,
            "regex_score": finding.regex_score,
            "severity": finding.severity,
            "evidence": finding.evidence,
        })
    return rows


In [ ]:
analyses = []
rows = []
metadata_rows = []

for _, case in test_cases.iterrows():
    analysis = run_detector(case["text"], case["case_id"])
    analyses.append(analysis)
    rows.extend(findings_to_rows(case, analysis))
    metadata_rows.append({
        "case_id": case["case_id"],
        "control_family": case["control_family"],
        "comment": case["comment"],
        "enabled_branches": " | ".join(analysis.metadata.get("enabled_branches", [])),
        "finding_count": analysis.metadata.get("finding_count"),
        "generic_finding_count": analysis.metadata.get("generic_finding_count"),
        "spacy_finding_count": analysis.metadata.get("spacy_finding_count"),
        "gliner_finding_count": analysis.metadata.get("gliner_finding_count"),
        "regex_finding_count": analysis.metadata.get("regex_finding_count"),
        "generic_max_score": analysis.metadata.get("generic_max_score"),
        "spacy_max_score": analysis.metadata.get("spacy_max_score"),
        "gliner_max_score": analysis.metadata.get("gliner_max_score"),
        "regex_max_score": analysis.metadata.get("regex_max_score"),
        "branch_errors": analysis.metadata.get("branch_errors"),
    })

results_df = pd.DataFrame(rows)
case_summary_df = pd.DataFrame(metadata_rows)
case_summary_df


In [ ]:
alerts_df = results_df[results_df["predicted_key"].notna()].copy()
alerts_df.sort_values(["case_id", "detection_engine", "score"], ascending=[True, True, False])


In [ ]:
comparison_df = (
    alerts_df.groupby(["case_id", "control_family", "comment", "detection_engine"], dropna=False)
    .agg(
        detection_count=("predicted_key", "size"),
        max_score=("score", "max"),
        predicted_keys=("predicted_key", lambda values: " | ".join(sorted({value for value in values if value}))),
        matched_terms=("matched_term", lambda values: " | ".join(sorted({value for value in values if value}))),
    )
    .reset_index()
)
comparison_df


In [ ]:
count_pivot_df = comparison_df.pivot_table(
    index=["case_id", "control_family", "comment"],
    columns="detection_engine",
    values="detection_count",
    aggfunc="sum",
    fill_value=0,
).reset_index()
count_pivot_df


In [ ]:
score_columns = ["generic_score", "spacy_score", "gliner_score", "regex_score"]
score_view_df = alerts_df[[
    "case_id",
    "detection_engine",
    "predicted_key",
    "matched_term",
    "detection_type",
    *score_columns,
    "evidence",
]].sort_values(["case_id", "detection_engine", "predicted_key"], na_position="last")
score_view_df


In [ ]:
branch_errors_df = case_summary_df[["case_id", "branch_errors"]].copy()
branch_errors_df = branch_errors_df[branch_errors_df["branch_errors"].map(bool)]
branch_errors_df
